In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [2]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    print(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        print("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    print("No valid indicator data found.")
    observed_src = pd.DataFrame()


Querying owner: HTOC Org


In [21]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,legacyLink,tags.data,associatedGroups.data,hostName,dnsActive,whoisActive,source,address,url,indicator
0,5629499554313260,2025-06-11T14:46:28Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-21T13:38:17Z,3.0,75,RITM0588707 / TASK0886419,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,176.65.149.160
1,6755399460007896,2025-07-02T12:05:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-21T13:24:58Z,4.0,71,"Details\nIn late June 2025, Iran-aligned hackt...",...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 1469320, 'name': 'INDICATOR NOTICE 251...","[{'id': 5629499544001057, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,49.232.59.192
2,6755399447111266,2025-05-14T17:47:38Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-21T13:24:44Z,3.0,71,FBI Email Alert May 14 2025 Medium IP IOCs,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,61.132.217.130
3,5629499559400773,2025-07-15T15:07:04Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-21T13:24:40Z,3.0,84,NaN,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...","[{'id': 6755399453001053, 'dateAdded': '2025-0...",yourpensionmeeting.com,False,False,NaN,NaN,NaN,yourpensionmeeting.com
4,6755399448251600,2025-05-30T13:30:26Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-21T12:42:14Z,1.0,74,CMS received a volumetric alert for thirty six...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 471298, 'name': 'DHA Splunk API', 'las...","[{'id': 6755399448004487, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,NaN,89.248.163.54
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1038,4883606,2024-09-09T18:29:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-16T23:23:56Z,3.0,76,VA completed an investigation of a potential p...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 38829, 'name': 'Phishing', 'descriptio...","[{'id': 455288, 'dateAdded': '2024-09-09T18:25...",NaN,NaN,NaN,https://clickme.thryv.com,NaN,clickme.thryv.com,clickme.thryv.com
1039,4588250,2024-05-02T20:34:24Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-05-02T20:37:13Z,3.0,70,Actors often use defunct online text editors t...,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 261007, 'name': 'Abuse of Functionalit...","[{'id': 333596, 'dateAdded': '2024-05-02T20:31...",NaN,NaN,NaN,NaN,NaN,fckeditor.net,fckeditor.net
1040,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83,NaN,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 35760, 'name': 'OS Splunk API', 'descr...",NaN,NaN,NaN,NaN,https://google,NaN,google,google
1041,4491603,2023-12-21T16:13:01Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2023-12-22T11:56:27Z,3.0,89,"On December 11, 2023, VA users received a susp...",...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 38829, 'name': 'Phishing', 'descriptio...","[{'id': 292886, 'dateAdded': '2023-12-21T16:12...",NaN,NaN,NaN,https://financier.com,NaN,financier.com,financier.com


In [38]:
import pandas as pd

df = observed_src.copy()

# --- FULL TAGS UNNEST + PARTNERS ---

# explode tags.data
tags_exploded = (
    df[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
    # now tags_exploded['tags.data'] is always a dict
)

# normalize all fields in tags.data into flat columns
tags_norm = pd.json_normalize(tags_exploded['tags.data'])
# prefix so we don’t collide with other columns
tags_norm.columns = [f"tag_{c}" for c in tags_norm.columns]

# re‑attach indicator
tags_expanded = tags_exploded.reset_index(drop=True).join(tags_norm)

# extract partners as before (you can drop this if you only want full fields)
tags_expanded['partner'] = tags_expanded['tag_name'].map(
    lambda n: n[:-len(' Splunk API')]
    if isinstance(n, str) and n.endswith(' Splunk API')
    else None
)

# aggregate each tag_* field into a list per indicator
tag_fields = [c for c in tags_norm.columns]
tag_agg = (
    tags_expanded
    .groupby('indicator', as_index=False)
    .agg({ **{f: list for f in tag_fields},
           'partner':  lambda x: [p for p in set(x) if p] })
)

# rename partner → partners
tag_agg = tag_agg.rename(columns={'partner':'partners'})


# --- GROUPS VIA EXPLODE + GROUPBY ---

groups_exploded = (
    df[['indicator', 'associatedGroups.data']]
      .explode('associatedGroups.data')
      .dropna(subset=['associatedGroups.data'])
)

# normalize dict into columns
group_norm = pd.json_normalize(
    groups_exploded['associatedGroups.data']
).rename(columns={'id':'group_id','name':'group_name'})

groups_exploded = groups_exploded.reset_index(drop=True).join(group_norm)

# dedupe & aggregate ids/names
group_agg = (
    groups_exploded
    .drop_duplicates(subset=['indicator','group_id','group_name'])
    .groupby('indicator', as_index=False)
    .agg({
        'group_id':   lambda ids: list(ids),
        'group_name': lambda names: list(names)
    })
    .rename(columns={'group_id':'group_ids','group_name':'group_names'})
)


# --- CORE AGGREGATION OF OTHER COLUMNS ---

first_cols = [
    'id','dateAdded','ownerId','ownerName','webLink','type','lastModified',
    'rating','confidence','description','summary','observations',
    'lastObserved','privateFlag','active','activeLocked','ip',
    'legacyLink','hostName','dnsActive','whoisActive','source',
    'address','url'
]

base_agg = (
    df.drop(columns=[
        'createdBy.id', 'createdBy.userName', 'createdBy.firstName', 'createdBy.lastName',
        'createdBy.pseudonym', 'createdBy.owner', 'xid', 'eventType', 'documentDateAdded',
        'documentType', 'fileSize', 'fileName', 'downVoteCount', 'upVoteCount', 'type_group',
        'webLink_group', 'ownerName_group', 'ownerId_group', 'dateAdded_group', 'id_group',
        'platforms.count', 'tactics.count',
    ], errors='ignore')
    .groupby('indicator', as_index=False)[
        [c for c in first_cols if c not in ['address', 'ip', 'source', 'url', 'legacyLink']]
    ]
    .first()
)


# --- MERGE EVERYTHING BACK ---

agg_df = (
    base_agg
      .merge(tag_agg,     on='indicator', how='left')
      .merge(group_agg,   on='indicator', how='left')
)

# ensure list‑columns are lists
for col in ['partners','group_ids','group_names'] + tag_fields:
    agg_df[col] = agg_df[col].apply(lambda x: x if isinstance(x, list) else [])

display(agg_df)


,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,...,tag_lastUsed,tag_description,tag_techniqueId,tag_tactics.data,tag_tactics.count,tag_platforms.data,tag_platforms.count,partners,group_ids,group_names
0,101.89.174.236,5629499542017370,2025-05-21T18:39:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-15T07:55:12Z,3.0,73,...,"[2025-07-21T13:24:40Z, 2025-07-21T08:34:01Z, 2...","[nan, Observations reported by the HHS Ofice o...","[nan, nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan, nan]","[OS, IHS, DHA, FDA, CMS, NIH, HRSA]",[],[]
1,103.114.96.246,6755399460007420,2025-07-02T12:05:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-18T13:40:33Z,4.0,71,...,"[2025-07-18T13:37:52Z, 2025-07-18T13:37:52Z, 2...","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]",[],[5629499544001057],[INDICATOR NOTICE 25178.1 – Iran-Aligned Hackt...
2,103.125.189.6,5629499556005919,2025-06-30T17:15:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-21T07:36:47Z,3.0,77,...,"[2025-07-21T13:24:40Z, 2025-07-21T08:34:01Z, 2...","[nan, Observations reported by the HHS Ofice o...","[nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan]","[OS, CDC, DHA, FDA, CMS]",[],[]
3,103.131.196.158,5629499536034708,2025-04-11T17:47:39Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-18T22:43:51Z,4.0,78,...,"[2025-07-18T13:34:26Z, 2025-06-25T14:32:36Z, 2...","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan]",[],[6755399443004631],[Alert to Federal CISOs: IOCs associated with ...
4,103.133.60.12,6755399460008266,2025-07-02T12:05:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-07-21T02:53:35Z,4.0,71,...,"[2025-07-18T13:37:52Z, 2025-07-18T13:37:52Z, 2...","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]",[],[5629499544001057],[INDICATOR NOTICE 25178.1 – Iran-Aligned Hackt...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1036,www.deepseek.com,5271608,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-15T07:56:06Z,3.0,86,...,"[2025-07-21T13:24:40Z, 2025-01-31T14:04:11Z, 2...","[nan, nan, Observations reported by the HHS Of...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[OS, IHS, DHA, FDA, CMS, NIH, HRSA]","[548890, 548570]","[HTOC-20250131-0903-A, Blocked URLs by NIH for..."
1037,www.deepseek.com.cdn.cloudflare.net,5271612,2025-01-29T16:27:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-07-16T07:35:09Z,3.0,84,...,"[2025-07-21T13:24:40Z, 2025-01-31T14:04:11Z, 2...","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]","[nan, nan, nan, nan, nan, nan, nan, nan]","[IHS, DHA, CMS, NIH]"

In [ ]:
# Enrich only final filtered indicators
indicator_values = recent_tags['summary'].dropna().unique().tolist()
enriched_results = []

print(f"Enriching {len(indicator_values)} indicators with DomainTools and VirusTotalV3...")

for value in indicator_values:
    try:
        # Use the indicator *value*, not the ID
        encoded_value = urllib.parse.quote(value)
        enrich_url = f'/v3/indicators/{encoded_value}/enrich?type=Shodan&type=VirusTotalV3'
        ro.set_http_method('POST')
        ro.set_request_uri(enrich_url)
        ro.set_body({})
        enrich_response = tc.api_request(ro)

        if enrich_response.status_code == 200:
            enrich_data = enrich_response.json()
            enrich_data['summary'] = value  # Save the value as key
            enriched_results.append(enrich_data)
    except Exception as e:
        continue

if enriched_results:
    df_enriched = pd.json_normalize(enriched_results)
    df_enriched = pd.json_normalize(enriched_results)
    recent_tags = recent_tags.merge(df_enriched, on='summary', how='left')
    # Unnest 'data.enrichment.data' (list of dicts) into separate columns
if 'data.enrichment.data' in recent_tags.columns:
    enrichment_df = pd.json_normalize(
        recent_tags['data.enrichment.data'].dropna().explode()
    )
    enrichment_df.index = recent_tags['data.enrichment.data'].dropna().explode().index
    enrichment_cols = [col for col in enrichment_df.columns if col not in recent_tags.columns]
    # Join enrichment columns back to recent_tags
    recent_tags = recent_tags.join(enrichment_df[enrichment_cols], how='left')

    print(f"Successfully enriched and merged {len(df_enriched)} indicators.")
else:
    print("No enrichment data retrieved.")


Enriching 535 indicators with DomainTools and VirusTotalV3...


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host cdcfoundation.org cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host link.mail.beehiiv.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host congressionalaquarium.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host com.ar cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Host uploads-ssl.webflow.com cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed

Successfully enriched and merged 516 indicators.


In [ ]:
import numpy as np

# Unnest the 'data.enrichment.data' column into separate columns for each enrichment type

def extract_enrichment(row):
    """Extracts enrichment fields from the list of dicts in 'data.enrichment.data'."""
    enrichments = row.get('data.enrichment.data')
    result = {}
    if isinstance(enrichments, list):
        for enrich in enrichments:
            enrich_type = enrich.get('type')
            if enrich_type == 'Shodan':
                # Flatten Shodan fields
                for key in ['hostNames', 'domains', 'tags', 'country', 'city', 'isp', 'asn', 'org', 'openPorts']:
                    result[f'shodan_{key}'] = enrich.get(key, np.nan)
            elif enrich_type == 'VirusTotal':
                result['vtMaliciousCount'] = enrich.get('vtMaliciousCount', np.nan)
    return pd.Series(result)

# Apply extraction to recent_tags
enrichment_expanded = recent_tags.apply(extract_enrichment, axis=1)
recent_tags = pd.concat([recent_tags, enrichment_expanded], axis=1)

recent_tags = recent_tags.rename(columns={
    'indicator': 'Indicator',
    'vtMaliciousCount': 'Malicious Score/Count',
    'obs_date': 'Observation Date',
    'shodan_asn': 'ASN',
    'rating': 'ThreatAssessRating',
    'confidence': 'ThreatAssessConfidence',
    'shodan_city': 'City',
    'shodan_country': 'Country',
    'data.legacyLink': 'ThreatConnect Link',
    'partners': 'Partners'
})

# Now select only the columns you want, after renaming
recent_tags = recent_tags[
    [
        'Indicator',
        'Malicious Score/Count',
        'Observation Date',
        'ASN',
        'ThreatAssessRating',
        'ThreatAssessConfidence',
        'City',
        'Country',
        'ThreatConnect Link',
        'Partners'
    ]
]

# Remove duplicate columns by keeping only the first occurrence
recent_tags = recent_tags.loc[:, ~recent_tags.columns.duplicated()]
recent_tags = recent_tags.fillna('unknown')
recent_tags = recent_tags.drop_duplicates()
recent_tags

,Indicator,Malicious Score/Count,Observation Date,ASN,ThreatAssessRating,ThreatAssessConfidence,City,Country,ThreatConnect Link,Partners
0,65.49.1.127,unknown,2025-07-14,AS6939,1.0,3,San Francisco,United States,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, IHS, NIH"
0,65.49.1.127,6.0,2025-07-14,AS6939,1.0,3,San Francisco,United States,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, IHS, NIH"
1,193.163.125.118,unknown,2025-07-14,AS211298,3.0,77,Manchester,United Kingdom,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, IHS, NIH, OS"
1,193.163.125.118,7.0,2025-07-14,AS211298,3.0,77,Manchester,United Kingdom,https://hvs.threatconnect.com/auth/indicators/...,"CMS, FDA, IHS, NIH, OS"
2,109.71.252.97,unknown,2025-07-14,AS58087,4.0,72,Frankfurt am Main,Germany,https://hvs.threatconnect.com/auth/indicators/...,CMS
...,...,...,...,...,...,...,...,...,...,...
531,ajxx98.online,unknown,2025-07-14,unknown,3.0,75,unknown,unknown,unknown,DHA
532,34.160.111.145,unknown,2025-07-14,AS396982,1.0,0,Kansas City,United States,https://hvs.threatconnect.com/auth/indicators/...,CDC
532,34.160.111.145,2.0,2025-07-14,AS396982,1.0,0,Kansas City,United States,https://hvs.threatconnect.com/auth/indicators/...,CDC
533,www.shorturl.at/,unknown,2025-07-14,unknown,3.0,90,unknown,unknown,unknown,CDC


In [ ]:
enrichment_expanded

,shodan_asn,shodan_city,shodan_country,shodan_domains,shodan_hostNames,shodan_isp,shodan_openPorts,shodan_org,shodan_tags,vtMaliciousCount
0,AS396527,Cambridge,United States,NaN,NaN,Massachusetts Institute of Technology,"[{'transport': 'tcp', 'port': 2222, 'product':...",MIT,[tor],8.0
0,AS396527,Cambridge,United States,NaN,NaN,Massachusetts Institute of Technology,"[{'transport': 'tcp', 'port': 2222, 'product':...",MIT,[tor],8.0
1,AS30633,Washington,United States,NaN,NaN,"Leaseweb USA, Inc.","[{'transport': 'tcp', 'port': 80, 'product': '...","Leaseweb USA, Inc.",[tor],6.0
1,AS30633,Washington,United States,NaN,NaN,"Leaseweb USA, Inc.","[{'transport': 'tcp', 'port': 80, 'product': '...","Leaseweb USA, Inc.",[tor],6.0
2,AS64289,Amsterdam,Netherlands,NaN,NaN,Macarne.com,"[{'transport': 'tcp', 'port': 22, 'product': '...",Macarne Limited,NaN,5.0
...,...,...,...,...,...,...,...,...,...,...
593,AS20940,Ashburn,United States,"[akamaitechnologies.com, akamai.net]","[a248.e.akamai.net, a23-205-105-180.deploy.sta...",Akamai International B.V.,"[{'transport': 'tcp', 'port': 80, 'product': '...","Akamai Technologies, Inc.",[cdn],0.0
593,AS20940,Ashburn,United States,"[akamaitechnologies.com, akamai.net]","[a248.e.akamai.net, a23-205-105-180.deploy.sta...",Akamai International B.V.,"[{'transport': 'tcp', 'port': 80, 'product': '...","Akamai Technologies, Inc.",[cdn],0.0
594,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
595,AS396982,Kansas City,United States,"[ipv6addr.com, onlyip6.me, ip6.me, whatismyv6....","[145.111.160.34.bc.googleusercontent.com, ipv6...",Google LLC,"[{'transport': 'tcp', 'port': 80, 'data': 'HTT...",Google LLC,[cloud],2.0
